In [1]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

from sentence_transformers import SentenceTransformer

# Awalan

In [ ]:
df_cv = pd.read_csv('sample_10_cv_with_ground_truth.csv')
df_job = pd.read_csv('data_clean.csv')

In [4]:
df_cv.head()

,candidate_name,email,phone,skills,summary,experience,degree,university,Category,text_for_tfidf,text_for_embed,text_for_embed_with_punctuation,ground_truth
0,Rina Putri,rina.putri@example.com,6281234567890,"kepemimpinan, google workspace, kerja tim, mic...",dokter umum yang berdedikasi dengan pengalaman...,lebih dari 3 tahun di pelayanan kesehatan pri...,S1,universitas indonesia jakarta lulus dengan pre...,Healthcare,summary dedicated general practitioner 3 years...,summary dedicated general practitioner with mo...,summary: dedicated general practitioner with m...,"dokter umum, nurse, medical representative"
1,Siti Rahayu,siti.rahayu@email.com,62812345678900,asisten keperawatan magang februari 2023 - mei...,-,"dalam memberikan perawatan dasar, mendampingi ...",-,-,Healthcare,summary experience providing basic care accomp...,summary experience in providing basic care acc...,"summary: experience: in providing basic care, ...","perawat, staff nurse (perawat), private homeca..."
2,Rina Hidayati,rina.hidayati@email.com,6281123456789,"lorem ipsum corp jakarta, software sertifikasi...",seorang profesional call center dengan pengala...,lebih dari 2 tahun dalam menangani keluhan pe...,SMA,-,Customer Service,summary call center professional 2 years exper...,summary a call center professional with more t...,summary: a call center professional with more ...,"customer service, customer relationship office..."
3,Rina Sari,rina.sari@example.com,6281234567890,"kepemimpinan, matlab, excel, perancangan prose...",-,"dalam penelitian, analisis data, dan perancang...",S1,universitas pendidikan indonesia bandung ipk,Data,summary experience research data analysis proc...,summary experience in research data analysis a...,"summary: experience: in research, data analysi...",data analyst for production staff (code : prod...
4,Rizal Alfian,rizal.alfian@example.com,6281234567890,"hmt elektro upi bandung, autocad electrical, r...",-,"dalam desain sirkuit dan pengujian perangkat, ...",S1,universitas pendidikan indonesia bandung ipk,Engineering,summary experience circuit design device testi...,summary experience in circuit design and devic...,summary: experience: in circuit design and dev...,"engineering staff, electrical engineering (kos..."


In [5]:
df_job.head()

,Posisi,Perusahaan,Lokasi,Type,Gaji,Requirements,Link,text_for_tfidf,text_for_embed,text_for_embed_with_punctuation
0,Tenaga Admin,PT Serdang Baja Makmur Abadi,Medan,Full time,Rp 3.250.000 – Rp 3.500.000 per month,Pekerjaan administrasi umum seperti membuat In...,https://id.jobstreet.com/id/job/91338594?type=...,admin staff admin staff general administrative...,admin staff general administrative work such a...,admin staff general administrative work such a...
1,Admin Operasional & HR Support (MAGANG),PT Saptakarsa Prima,Kecamatan Tangerang,Full time,Rp 2.500.000 – Rp 3.500.000 per month,Pendidikan minimal SMK Jurusan Administrasi Pe...,https://id.jobstreet.com/id/job/91309622?type=...,admin operational hr support internship admin ...,admin operational hr support internship educat...,admin operational hr support internship educat...
2,Executive admin,Pengiklan Anonim,Jakarta Utara,Full time,Rp 4.700.000 – Rp 5.500.000 per month,Mengelola kalender dan jadwal eksekutif dengan...,https://id.jobstreet.com/id/job/91479903?type=...,executive admin executive admin manages execut...,executive admin manages executive calendars an...,executive admin manages executive calendars an...
3,Executive admin,Pengiklan Anonim,Jakarta Utara,Full time,Rp 4.700.000 – Rp 5.500.000 per month,"Mengelola jadwal, pertemuan, dan kegiatan ekse...",https://id.jobstreet.com/id/job/91508884?type=...,executive admin executive admin manages meetin...,executive admin manages meeting schedules and ...,executive admin manages meeting schedules and ...
4,Operations & Admin Coordinator (Indonesia Repr...,Pengiklan Anonim,Cilandak,Full time,Rp 6.500.000 – Rp 8.500.000 per month,Manage daily administrative tasks; Organize do...,https://id.jobstreet.com/id/job/91487019?type=...,operations admin coordinator indonesia represe...,operations admin coordinator indonesia represe...,operations admin coordinator indonesia represe...


In [6]:
df_cv.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 13 columns):
 #   Column                           Non-Null Count  Dtype 
---  ------                           --------------  ----- 
 0   candidate_name                   10 non-null     object
 1   email                            10 non-null     object
 2   phone                            10 non-null     int64 
 3   skills                           10 non-null     object
 4   summary                          10 non-null     object
 5   experience                       10 non-null     object
 6   degree                           10 non-null     object
 7   university                       10 non-null     object
 8   Category                         10 non-null     object
 9   text_for_tfidf                   10 non-null     object
 10  text_for_embed                   10 non-null     object
 11  text_for_embed_with_punctuation  10 non-null     object
 12  ground_truth                     10 non

In [7]:
df_job.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1400 entries, 0 to 1399
Data columns (total 10 columns):
 #   Column                           Non-Null Count  Dtype 
---  ------                           --------------  ----- 
 0   Posisi                           1400 non-null   object
 1   Perusahaan                       1400 non-null   object
 2   Lokasi                           1400 non-null   object
 3   Type                             1400 non-null   object
 4   Gaji                             1400 non-null   object
 5   Requirements                     1400 non-null   object
 6   Link                             1400 non-null   object
 7   text_for_tfidf                   1400 non-null   object
 8   text_for_embed                   1400 non-null   object
 9   text_for_embed_with_punctuation  1400 non-null   object
dtypes: object(10)
memory usage: 109.5+ KB


In [8]:
# df_cv_clean = df_cv.drop_duplicates(subset=['candidate_name'])
# df_cv_undersampling = df_cv_clean.sample(n=10, random_state=42).reset_index(drop=True)
# df_cv_undersampling

In [9]:
# df_cv_undersampling.to_csv('sample_10_cv.csv', index=False)

# TF-IDF

In [10]:
job_texts = df_job["text_for_tfidf"].tolist()

In [11]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),
    sublinear_tf=True
)

job_tfidf = vectorizer.fit_transform(job_texts)

In [12]:
for h in range(10):
    print("-"*40)
    print(f"BARIS {h+1}")
    print("-"*40)
    cv_text = df_cv.loc[h, 'text_for_tfidf']
    cv_tfidf = vectorizer.transform([cv_text])

    scores = cosine_similarity(cv_tfidf, job_tfidf)

    top_k = 5
    top_indices = scores[0].argsort()[-top_k:][::-1]

    for i in top_indices:
        print("Perusahaan:", df_job.iloc[i]["Perusahaan"])
        print("Posisi:", df_job.iloc[i]["Posisi"])
        print("Score:", scores[0][i])
        print("-"*40)

    print("\n")

----------------------------------------
BARIS 1
----------------------------------------
Perusahaan: Serrebeauty
Posisi: Dokter Umum
Score: 0.2805980835611086
----------------------------------------
Perusahaan: RS Awal Bros Group
Posisi: Kepala Departemen Penunjang Medis
Score: 0.27447319106128015
----------------------------------------
Perusahaan: Pengiklan Anonim
Posisi: Private Homecare Nurse
Score: 0.2727134085762191
----------------------------------------
Perusahaan: RS Firdaus
Posisi: Perawat
Score: 0.24000719734249729
----------------------------------------
Perusahaan: PT. Meimo Toronta Sera
Posisi: HOSPITAL MANAGER ( Pre Opening)
Score: 0.22250251831881596
----------------------------------------


----------------------------------------
BARIS 2
----------------------------------------
Perusahaan: Pengiklan Anonim
Posisi: Private Homecare Nurse
Score: 0.3616282862230051
----------------------------------------
Perusahaan: RS Firdaus
Posisi: Perawat
Score: 0.25756608059456

In [13]:
for no, ground_truth in enumerate(df_cv['ground_truth'].to_list(), start=1):
    print(no, ground_truth)

1 dokter umum, nurse, medical representative
2 perawat, staff nurse (perawat), private homecare nurse
3 customer service, customer relationship officer, call center officer
4 data analyst for production staff (code : prod da), pm data, jr. specialist data stewards
5 engineering staff, electrical engineering (kosmetik & farmasi), it support
6 teacher / teaching assistant / admin, ict officer, guru matematika sma
7 driver, driver pribadi, kordinator driver
8 data analyst for production staff (code : prod da), sales & marketing data analyst staf, pm data
9 sales executive, account officer / marketing, sales representative
10 arsitek, interior designer, drafter structural


In [14]:
# # manual
# baris1 = [1,0,1,1,0]
# baris2 = [1,1,1,1,0]
# baris3 = [1,1,0,1,1]
# baris4 = [0,0,0,0,0]
# baris5 = []
# baris6 = []
# baris7 = []
# baris8 = []
# baris9 = []
# baris10 = []

In [15]:
# data = pd.DataFrame({
#     'ground_truth': df_cv['ground_truth'].to_list(),
#     'hasil_prediksi': []
#     'relevan/tidak': [[] for _ in range(10)]
# })

In [16]:
# data

# TF-IDF + SVD

In [17]:
job_texts = df_job["text_for_tfidf"].tolist()

In [18]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),
    sublinear_tf=True
)

job_tfidf = vectorizer.fit_transform(job_texts)

In [19]:
svd = TruncatedSVD(n_components=100, random_state=42)

job_svd = svd.fit_transform(job_tfidf)

In [20]:
for h in range(10):
    print("-"*40)
    print(f"BARIS {h+1}")
    print("-"*40)
    cv_text = df_cv.loc[h, 'text_for_tfidf']
    cv_tfidf = vectorizer.transform([cv_text])
    cv_svd = svd.transform(cv_tfidf)

    scores_svd = cosine_similarity(cv_svd, job_svd)

    top_k = 5
    top_indices = scores_svd[0].argsort()[-top_k:][::-1]

    for i in top_indices:
        print("Perusahaan:", df_job.iloc[i]["Perusahaan"])
        print("Posisi:", df_job.iloc[i]["Posisi"])
        print("Score:", scores_svd[0][i])
        print("-"*40)

    print("\n")

----------------------------------------
BARIS 1
----------------------------------------
Perusahaan: PT GENOMIK SOLIDARITAS INDONESIA
Posisi: Perawat Perusahaan
Score: 0.7510799251188788
----------------------------------------
Perusahaan: Pengiklan Anonim
Posisi: Private Homecare Nurse
Score: 0.7279133242296282
----------------------------------------
Perusahaan: Serrebeauty
Posisi: Dokter Umum
Score: 0.7222331524767647
----------------------------------------
Perusahaan: Klinik Sakti Medika
Posisi: Perawat Bedah
Score: 0.7214502161703373
----------------------------------------
Perusahaan: EUROMEDICA GROUP
Posisi: perawat
Score: 0.7113861433700511
----------------------------------------


----------------------------------------
BARIS 2
----------------------------------------
Perusahaan: Pengiklan Anonim
Posisi: Private Homecare Nurse
Score: 0.7696247769620093
----------------------------------------
Perusahaan: Klinik Sakti Medika
Posisi: Perawat Bedah
Score: 0.7617207325170107
-

In [21]:
for no, ground_truth in enumerate(df_cv['ground_truth'].to_list(), start=1):
    print(no, ground_truth)

1 dokter umum, nurse, medical representative
2 perawat, staff nurse (perawat), private homecare nurse
3 customer service, customer relationship officer, call center officer
4 data analyst for production staff (code : prod da), pm data, jr. specialist data stewards
5 engineering staff, electrical engineering (kosmetik & farmasi), it support
6 teacher / teaching assistant / admin, ict officer, guru matematika sma
7 driver, driver pribadi, kordinator driver
8 data analyst for production staff (code : prod da), sales & marketing data analyst staf, pm data
9 sales executive, account officer / marketing, sales representative
10 arsitek, interior designer, drafter structural


# Embedding (with_punctuation)

In [22]:
job_texts = df_job["text_for_embed_with_punctuation"].tolist()

In [23]:
model = SentenceTransformer('all-MiniLM-L6-v2')
# model = SentenceTransformer('all-mpnet-base-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [24]:
job_embeddings = model.encode(job_texts, show_progress_bar=True)

Batches:   0%|          | 0/44 [00:00<?, ?it/s]

In [25]:
for h in range(10):
    print("-"*40)
    print(f"BARIS {h+1}")
    print("-"*40)
    cv_text = df_cv.loc[h, 'text_for_embed_with_punctuation']
    cv_embedding = model.encode([cv_text])

    scores_embed = cosine_similarity(cv_embedding, job_embeddings)

    top_k = 5
    top_indices = scores_embed[0].argsort()[-top_k:][::-1]

    for i in top_indices:
        print("Perusahaan:", df_job.iloc[i]["Perusahaan"])
        print("Posisi:", df_job.iloc[i]["Posisi"])
        print("Score:", scores_embed[0][i])
        print("-"*40)

    print("\n")

----------------------------------------
BARIS 1
----------------------------------------
Perusahaan: Serrebeauty
Posisi: Dokter Umum
Score: 0.7168473
----------------------------------------
Perusahaan: PT Transfarma Medica Indah
Posisi: Medical Representative (Surabaya, Jakarta, Balikpapan)
Score: 0.6527623
----------------------------------------
Perusahaan: PT Biolife Medilab
Posisi: Professional Medical Representative (Beberapa Kota)
Score: 0.63205194
----------------------------------------
Perusahaan: Nusa Medica Clinic
Posisi: Call Center Officer
Score: 0.62851584
----------------------------------------
Perusahaan: SCCR Indonesia
Posisi: Medical Representative
Score: 0.6051338
----------------------------------------


----------------------------------------
BARIS 2
----------------------------------------
Perusahaan: MAYAPADA HEALTHCARE
Posisi: HEAD OF MAYAPADA CLINIC
Score: 0.5217646
----------------------------------------
Perusahaan: Klinik Sakti Medika
Posisi: Perawat Be

In [26]:
for no, ground_truth in enumerate(df_cv['ground_truth'].to_list(), start=1):
    print(no, ground_truth)

1 dokter umum, nurse, medical representative
2 perawat, staff nurse (perawat), private homecare nurse
3 customer service, customer relationship officer, call center officer
4 data analyst for production staff (code : prod da), pm data, jr. specialist data stewards
5 engineering staff, electrical engineering (kosmetik & farmasi), it support
6 teacher / teaching assistant / admin, ict officer, guru matematika sma
7 driver, driver pribadi, kordinator driver
8 data analyst for production staff (code : prod da), sales & marketing data analyst staf, pm data
9 sales executive, account officer / marketing, sales representative
10 arsitek, interior designer, drafter structural
